#  AI-Powered AC Competitor Analysis using LLM

## Problem Statement
In today’s competitive market, customers and businesses often struggle to compare products across different brands due to scattered and unstructured information available on company websites.

This project aims to solve that problem by building an **AI-powered system** that automatically analyzes and compares Air Conditioners (ACs) from leading brands like Samsung and Panasonic using website data and a Large Language Model (LLM).

##  Objective

Develop a system that:
- Extracts product-related data from company websites  
- Identifies key features, pricing, and specifications  
- Uses an LLM to generate a structured comparison 


In [ ]:
#import
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

class ACComparator:
    def __init__(self, model="llama3.2:1b"):
        self.model = model
        self.headers = {
            "User-Agent": "Mozilla/5.0"
        }

    def smart_fetch(self, url):
        try:
            res = requests.get(url, headers=self.headers, timeout=10)
            return res.text
        except:
            return ""
    
    def get_text(self, url):
        html = self.smart_fetch(url)
        soup = BeautifulSoup(html, "html.parser")

        for tag in soup(["script", "style"]):
            tag.decompose()

        return soup.get_text(separator="\n", strip=True)
    
    def call_llm(self, prompt, system_prompt="You are a helpful assistant"):
        from openai import OpenAI

        client = OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )

        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content
    
    def generate_AC_Comparison(self, name1, data1, name2, data2):
        prompt = f"""
        Compare {name1} and {name2} Air Conditioners.

        DATA {name1}:
        {data1[:3000]}

        DATA {name2}:
        {data2[:3000]}

        Generate:
        1. Comparison table (Price, Features, Energy Efficiency)
        2. Strengths of each
        3. Which is better for Indian users
        4. Final recommendation
        """

        return self.call_llm(prompt)

In [ ]:
analyst = ACComparator()

# AC Pages
samsung_url = "https://www.samsung.com/in/air-conditioners/"
panasonic_url = "https://store.in.panasonic.com/air-conditioners.html"

# Fetch content
samsung_data = analyst.get_text(samsung_url)
panasonic_data = analyst.get_text(panasonic_url)

# Generate comparison
result = analyst.generate_AC_Comparison(
    "Samsung",
    samsung_data,
    "Panasonic",
    panasonic_data
)

print(result)

## Conclusion

This project demonstrates how LLMs can be combined with web scraping to transform unstructured web data into meaningful business insights, enabling smarter and faster decision-making.

